# 01. Build the database

Creates four tables in `medicare.duckdb` from the raw files:

| Table | What it is |
|---|---|
| `raw` | the full CMS Medicare Geographic Variation file, 2014-2024 (national, state, county) |
| `county` | county rows for all ages, key columns renamed and converted to numbers |
| `analysis` | counties with at least 500 beneficiaries and non-missing spending, territories excluded |
| `saipe` | Census median household income and poverty rate by county and year |

**Before running:**
1. Download the CMS file into `data/2014-2024-Medicare-data.csv` (see the README).
2. Put your Census API key in a file called `.env` in the project folder:
   `CENSUS_API_KEY=your_key_here`. The `.gitignore` keeps that file off GitHub.

Every table is rebuilt from source on each run (`CREATE OR REPLACE`).

In [ ]:
import os

import duckdb
import pandas as pd
import requests
from dotenv import load_dotenv

con = duckdb.connect("../medicare.duckdb")

## 1. Load the CMS file

Hidden small-county values are `*` and missing values are `NA`. `nullstr='NA'`
turns `NA` into a real missing value. The `*` values are handled in step 2.

In [ ]:
con.sql("""
CREATE OR REPLACE TABLE raw AS
SELECT * FROM read_csv('../data/2014-2024-Medicare-data.csv', nullstr = 'NA')
""")

con.sql("SELECT COUNT(*) AS rows FROM raw")

In [ ]:
# Rows per year and geography level (all ages)
con.sql("""
SELECT YEAR, BENE_GEO_LVL, COUNT(*) AS n
FROM raw
WHERE BENE_AGE_LVL = 'All'
GROUP BY ALL
ORDER BY YEAR, BENE_GEO_LVL
""").show(max_rows=50)

## 2. County table

Because some cells contain `*`, DuckDB reads those columns as text.
`TRY_CAST` converts each one to a number and turns anything that is not a
number (the `*` values) into a missing value instead of raising an error.

In [ ]:
# How many suppressed values are there in the spending column?
con.sql("""
SELECT TOT_MDCR_STDZD_PYMT_PC AS value, COUNT(*) AS n
FROM raw
WHERE BENE_GEO_LVL = 'County' AND BENE_AGE_LVL = 'All'
  AND TRY_CAST(TOT_MDCR_STDZD_PYMT_PC AS DOUBLE) IS NULL
GROUP BY 1
""")

In [ ]:
con.sql("""
CREATE OR REPLACE TABLE county AS
SELECT
    YEAR                                              AS year,
    BENE_GEO_CD                                       AS fips,
    BENE_GEO_DESC                                     AS county_name,
    TRY_CAST(BENES_OM_CNT AS INTEGER)                 AS benes,
    TRY_CAST(TOT_MDCR_STDZD_PYMT_PC AS DOUBLE)        AS spend_pc_std,
    TRY_CAST(TOT_MDCR_PYMT_PC AS DOUBLE)              AS spend_pc_actual,
    TRY_CAST(BENE_AVG_AGE AS DOUBLE)                  AS avg_age,
    TRY_CAST(BENE_DUAL_PCT AS DOUBLE)                 AS dual_pct,
    TRY_CAST(MA_PRTCPTN_RATE AS DOUBLE)               AS ma_rate,
    TRY_CAST(ER_VISITS_PER_1000_BENES AS DOUBLE)      AS er_per_1000,
    TRY_CAST(IP_CVRD_STAYS_PER_1000_BENES AS DOUBLE)  AS ip_stays_per_1000,
    TRY_CAST(ACUTE_HOSP_READMSN_PCT AS DOUBLE)        AS readmit_pct
FROM raw
WHERE BENE_GEO_LVL = 'County'
  AND BENE_AGE_LVL = 'All'
""")

con.sql("""
SELECT COUNT(*)                                     AS total_rows,
       COUNT(*) FILTER (WHERE spend_pc_std IS NULL) AS missing_spend,
       COUNT(*) FILTER (WHERE benes < 500)          AS under_500_benes
FROM county
""")

## 3. Why a 500-beneficiary minimum?

The extremes in the full county table are dominated by tiny counties, where a
handful of people determine the average.

In [ ]:
con.sql("""
(SELECT 'lowest' AS which, year, county_name, benes, spend_pc_std
 FROM county WHERE spend_pc_std IS NOT NULL
 ORDER BY spend_pc_std ASC LIMIT 5)
UNION ALL
(SELECT 'highest', year, county_name, benes, spend_pc_std
 FROM county WHERE spend_pc_std IS NOT NULL
 ORDER BY spend_pc_std DESC LIMIT 5)
""")

## 4. Analysis sample

* spending not missing
* at least 500 Original Medicare beneficiaries
* the 50 states only (US territories excluded; DC has no county row in the CMS file)
* `state` is the two-letter prefix of `county_name` (e.g. `GA-Miller`)

In [ ]:
con.sql("""
CREATE OR REPLACE TABLE analysis AS
SELECT *,
       LEFT(county_name, 2) AS state
FROM county
WHERE spend_pc_std IS NOT NULL
  AND benes >= 500
  AND LEFT(county_name, 2) NOT IN ('PR', 'VI', 'GU', 'AS', 'MP')
""")

con.sql("""
SELECT COUNT(*)                 AS rows,
       COUNT(DISTINCT fips)     AS counties,
       COUNT(DISTINCT state)    AS states,
       ROUND(MIN(spend_pc_std)) AS min_spend,
       ROUND(MAX(spend_pc_std)) AS max_spend
FROM analysis
""")

## 5. Census SAIPE: income and poverty

Pulled from the Census API, one request per year. The key is read from `.env`
so it never appears in the code.

* `SAEMHI_PT`: median household income
* `SAEPOVRTALL_PT`: poverty rate, all ages (percent)

In [ ]:
load_dotenv("../.env")
CENSUS_KEY = os.getenv("CENSUS_API_KEY")
if not CENSUS_KEY:
    raise RuntimeError("No CENSUS_API_KEY found. Add it to a .env file in the project folder.")

frames = []
for yr in range(2014, 2025):
    r = requests.get(
        "https://api.census.gov/data/timeseries/poverty/saipe",
        params={"get": "NAME,SAEMHI_PT,SAEPOVRTALL_PT", "for": "county:*", "time": yr, "key": CENSUS_KEY},
        timeout=60,
    )
    if r.status_code != 200 or not r.text.startswith("["):
        # Print only the status, not r.url, which would contain the key.
        print(yr, "failed:", r.status_code)
        continue
    data = r.json()
    frames.append(pd.DataFrame(data[1:], columns=data[0]))
    print(yr, "ok,", len(data) - 1, "counties")

In [ ]:
saipe = pd.concat(frames, ignore_index=True)
saipe["fips"] = saipe["state"] + saipe["county"]   # 2-digit state + 3-digit county
saipe = saipe.rename(columns={"time": "year", "SAEMHI_PT": "median_income", "SAEPOVRTALL_PT": "poverty_rate"})
saipe["year"] = saipe["year"].astype(int)
saipe["median_income"] = pd.to_numeric(saipe["median_income"], errors="coerce")
saipe["poverty_rate"] = pd.to_numeric(saipe["poverty_rate"], errors="coerce")

# DuckDB can query a pandas DataFrame directly by its variable name.
con.sql("CREATE OR REPLACE TABLE saipe AS SELECT year, fips, NAME, median_income, poverty_rate FROM saipe")
con.sql("SELECT year, COUNT(*) AS counties FROM saipe GROUP BY 1 ORDER BY 1")

In [ ]:
con.close()  # release the file so the next notebook can open it